# Weather API project: Medallion architecture (Gold layer) using OpenMeteo API
### by Matias Bertuzzi
### Data Engineer | SunnyData

#1. Configuration

In [0]:
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.testing.utils import assertDataFrameEqual

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
catalog = f"mbertuzzi_{environment}"
dbutils.widgets.text("schema", "weatherapi", "Schema")
dbutils.widgets.text(
    "silver_table",
    "silver_weather_hourly",
    "Silver table name",
)
dbutils.widgets.text(
    "gold_table",
    "gold_weather_daily",
    "Gold table name",
)
dbutils.widgets.text("env", "dev", "Environment (dev/prod)")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
silver_table = dbutils.widgets.get("silver_table")
gold_table = dbutils.widgets.get("gold_table")
env = dbutils.widgets.get("env")

In [0]:
silver_table_fqn = f"{catalog}.{schema}.{silver_table}"
gold_table_fqn = f"{catalog}.{schema}.{gold_table}"

print(f"Source table: {silver_table_fqn}")
print(f"Target table: {gold_table_fqn}")

#2. Transformation

In [0]:
# COMMAND ----------

def build_daily_weather_summary(silver_df):
    """
    Aggregate the Silver hourly weather dataset to one row per
    location and forecast date.

    Grain:
        location_name + weather_date
    """

    daily_base = (
        silver_df
        .withColumn("weather_date", F.to_date("forecast_time"))
        .groupBy(
            "location_name",
            "latitude",
            "longitude",
            "weather_date",
        )
        .agg(
            F.min("forecast_time").alias("first_forecast_time"),
            F.max("forecast_time").alias("last_forecast_time"),

            F.count("*").alias("forecast_hours"),

            F.min("temperature_2m").alias("min_temperature_2m"),
            F.max("temperature_2m").alias("max_temperature_2m"),
            F.avg("temperature_2m").alias("avg_temperature_2m"),

            F.avg("relative_humidity_2m").alias(
                "avg_relative_humidity_2m"
            ),

            F.sum("precipitation_mm").alias(
                "total_precipitation_mm"
            ),

            F.sum(
                F.when(F.col("precipitation_mm") > 0, 1)
                .otherwise(0)
            ).alias("precipitation_hours"),

            F.max("_bronze_ingestion_timestamp").alias(
                "_silver_max_ingestion_timestamp"
            ),

            F.max("_source_url").alias("_source_url"),
        )
    )

    # Determine the dominant weather code for each location/day.
    weathercode_counts = (
        silver_df
        .withColumn("weather_date", F.to_date("forecast_time"))
        .groupBy(
            "location_name",
            "weather_date",
            "weathercode",
        )
        .agg(
            F.count("*").alias("weathercode_count")
        )
    )

    weathercode_window = Window.partitionBy(
        "location_name",
        "weather_date",
    ).orderBy(
        F.col("weathercode_count").desc(),
        F.col("weathercode").asc(),
    )

    dominant_weathercode = (
        weathercode_counts
        .withColumn(
            "rank",
            F.row_number().over(weathercode_window),
        )
        .filter(F.col("rank") == 1)
        .select(
            "location_name",
            "weather_date",
            F.col("weathercode").alias("dominant_weathercode"),
        )
    )

    return (
        daily_base
        .join(
            dominant_weathercode,
            on=["location_name", "weather_date"],
            how="left",
        )
        .select(
            "location_name",
            "latitude",
            "longitude",
            "weather_date",
            "first_forecast_time",
            "last_forecast_time",
            "forecast_hours",
            "min_temperature_2m",
            "max_temperature_2m",
            "avg_temperature_2m",
            "avg_relative_humidity_2m",
            "total_precipitation_mm",
            "precipitation_hours",
            "dominant_weathercode",
            "_silver_max_ingestion_timestamp",
            "_source_url",
        )
    )

In [0]:
from datetime import datetime

test_silver_df = spark.createDataFrame(
    [
        (
            "Buenos Aires",
            -34.6037,
            -58.3816,
            datetime(2026, 1, 1, 0, 0),
            20.0,
            70.0,
            0.0,
            1,
            datetime(2026, 1, 1, 0, 5),
            "https://fixture.test",
        ),
        (
            "Buenos Aires",
            -34.6037,
            -58.3816,
            datetime(2026, 1, 1, 1, 0),
            18.0,
            80.0,
            1.5,
            2,
            datetime(2026, 1, 1, 0, 5),
            "https://fixture.test",
        ),
        (
            "Buenos Aires",
            -34.6037,
            -58.3816,
            datetime(2026, 1, 1, 2, 0),
            19.0,
            75.0,
            0.5,
            2,
            datetime(2026, 1, 1, 0, 5),
            "https://fixture.test",
        ),
    ],
    schema="""
        location_name STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        forecast_time TIMESTAMP,
        temperature_2m DOUBLE,
        relative_humidity_2m DOUBLE,
        precipitation_mm DOUBLE,
        weathercode INT,
        _bronze_ingestion_timestamp TIMESTAMP,
        _source_url STRING
    """,
)

test_gold_df = build_daily_weather_summary(test_silver_df)

expected_gold_df = spark.createDataFrame(
    [
        (
            "Buenos Aires",
            -34.6037,
            -58.3816,
            datetime(2026, 1, 1).date(),
            datetime(2026, 1, 1, 0, 0),
            datetime(2026, 1, 1, 2, 0),
            3,
            18.0,
            20.0,
            19.0,
            75.0,
            2.0,
            2,
            2,
            datetime(2026, 1, 1, 0, 5),
            "https://fixture.test",
        )
    ],
    schema="""
        location_name STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        weather_date DATE,
        first_forecast_time TIMESTAMP,
        last_forecast_time TIMESTAMP,
        forecast_hours BIGINT,
        min_temperature_2m DOUBLE,
        max_temperature_2m DOUBLE,
        avg_temperature_2m DOUBLE,
        avg_relative_humidity_2m DOUBLE,
        total_precipitation_mm DOUBLE,
        precipitation_hours BIGINT,
        dominant_weathercode INT,
        _silver_max_ingestion_timestamp TIMESTAMP,
        _source_url STRING
    """,
)

assertDataFrameEqual(
    test_gold_df,
    expected_gold_df,
    ignoreColumnOrder=True,
)

print("build_daily_weather_summary() regression test passed")

In [0]:
silver_df = spark.table(silver_table_fqn)

gold_table_exists = spark.catalog.tableExists(gold_table_fqn)

watermark = None

if gold_table_exists:
    watermark = (
        spark.sql(
            f"""
            SELECT MAX(_silver_max_ingestion_timestamp) AS wm
            FROM {gold_table_fqn}
            """
        )
        .collect()[0]["wm"]
    )

print(f"Gold watermark: {watermark}")

In [0]:
if watermark is None:
    affected_silver_df = silver_df
else:
    affected_silver_df = silver_df.filter(
        F.col("_bronze_ingestion_timestamp") > F.lit(watermark)
    )

affected_dates_df = (
    affected_silver_df
    .select(
        "location_name",
        F.to_date("forecast_time").alias("weather_date"),
    )
    .distinct()
)

affected_dates_count = affected_dates_df.count()

print(f"Affected location/date combinations: {affected_dates_count}")

In [0]:
if affected_dates_count == 0:
    dbutils.notebook.exit(
        "No new Silver data to process — exiting Gold run."
    )

In [0]:
silver_with_date_df = (
    silver_df
    .withColumn(
        "weather_date",
        F.to_date("forecast_time"),
    )
)

affected_silver_df = (
    silver_with_date_df
    .join(
        affected_dates_df,
        on=["location_name", "weather_date"],
        how="inner",
    )
)

In [0]:
gold_df = build_daily_weather_summary(
    affected_silver_df
)

gold_df = gold_df.select(
    "location_name",
    "latitude",
    "longitude",
    "weather_date",
    "first_forecast_time",
    "last_forecast_time",
    "forecast_hours",
    "min_temperature_2m",
    "max_temperature_2m",
    "avg_temperature_2m",
    "avg_relative_humidity_2m",
    "total_precipitation_mm",
    "precipitation_hours",
    "dominant_weathercode",
    "_silver_max_ingestion_timestamp",
    "_source_url",
)

In [0]:
from pyspark.sql.types import DoubleType

double_cols = [
    field.name
    for field in gold_df.schema.fields
    if isinstance(field.dataType, DoubleType)
]

for col_name in double_cols:
    gold_df = gold_df.withColumn(col_name, F.round(F.col(col_name), 2))

#3. Write

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {gold_table_fqn} (
        location_name STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        weather_date DATE,
        first_forecast_time TIMESTAMP,
        last_forecast_time TIMESTAMP,
        forecast_hours BIGINT,
        min_temperature_2m DOUBLE,
        max_temperature_2m DOUBLE,
        avg_temperature_2m DOUBLE,
        avg_relative_humidity_2m DOUBLE,
        total_precipitation_mm DOUBLE,
        precipitation_hours BIGINT,
        dominant_weathercode INT,
        _silver_max_ingestion_timestamp TIMESTAMP,
        _source_url STRING
    )
    USING DELTA
    CLUSTER BY (weather_date, location_name)
    """
)

In [0]:
target = DeltaTable.forName(
    spark,
    gold_table_fqn,
)

(
    target.alias("t")
    .merge(
        gold_df.alias("s"),
        """
        t.location_name = s.location_name
        AND t.weather_date = s.weather_date
        """,
    )
    .whenMatchedUpdate(
        condition="""
            s._silver_max_ingestion_timestamp
            > t._silver_max_ingestion_timestamp
        """,
        set={
            "latitude": "s.latitude",
            "longitude": "s.longitude",
            "first_forecast_time": "s.first_forecast_time",
            "last_forecast_time": "s.last_forecast_time",
            "forecast_hours": "s.forecast_hours",
            "min_temperature_2m": "s.min_temperature_2m",
            "max_temperature_2m": "s.max_temperature_2m",
            "avg_temperature_2m": "s.avg_temperature_2m",
            "avg_relative_humidity_2m": (
                "s.avg_relative_humidity_2m"
            ),
            "total_precipitation_mm": (
                "s.total_precipitation_mm"
            ),
            "precipitation_hours": "s.precipitation_hours",
            "dominant_weathercode": (
                "s.dominant_weathercode"
            ),
            "_silver_max_ingestion_timestamp": (
                "s._silver_max_ingestion_timestamp"
            ),
            "_source_url": "s._source_url",
        },
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.sql(
    f"OPTIMIZE {gold_table_fqn}"
)

In [0]:
if env == "dev":
    spark.sql(
        f"""
        ALTER TABLE {gold_table_fqn}
        SET TBLPROPERTIES (
            'delta.deletedFileRetentionDuration' =
            'interval 24 hours'
        )
        """
    )
else:
    # Prod keeps Delta's default 168h (7-day)
    # time-travel safety window.
    pass

spark.sql(
    f"VACUUM {gold_table_fqn}"
)

In [0]:
#display(spark.table(gold_table_fqn))

In [0]:
duplicate_grain_count = (
    spark.table(gold_table_fqn)
    .groupBy(
        "location_name",
        "weather_date",
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_grain_count > 0:
    raise ValueError(
        f"Gold grain validation failed: "
        f"{duplicate_grain_count} duplicate "
        f"location/date combinations found."
    )

null_grain_count = (
    spark.table(gold_table_fqn)
    .filter(
        F.col("location_name").isNull()
        | F.col("weather_date").isNull()
    )
    .count()
)

if null_grain_count > 0:
    raise ValueError(
        f"Gold grain validation failed: "
        f"{null_grain_count} rows have null grain columns."
    )

print(
    f"Gold validation passed for {gold_table_fqn}"
)